# Classical TF-IDF baselines
Runs `scripts/10_run_classical_baselines.py` end-to-end on Google Colab.


## 1. Mount Drive & clone repo
Set `USE_DRIVE=True` to persist `data/`, `results/`, `figures/` across Colab runtime resets. 
Set `REPO_URL` to override the auto-detected git remote.


In [ ]:
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/CARA-FinSent"  #@param {type:"string"}
REPO_URL = ""  #@param {type:"string"}  # leave blank to auto-detect from this notebook's repo

import os, subprocess
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    WORK_DIR = Path(DRIVE_DIR)
else:
    WORK_DIR = Path("/content")

REPO_DIR = WORK_DIR / "cara-finsent-experiments"
if not REPO_DIR.exists():
    if not REPO_URL:
        # Best-effort auto-detect: try the repo this notebook lives in (works when notebook is opened from GitHub).
        REPO_URL = os.environ.get("REPO_URL", "")
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL above (e.g. https://github.com/<user>/cara-finsent-experiments.git)")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working in:", os.getcwd())


## 2. Install dependencies


In [ ]:
!pip -q install -r requirements.txt


## 3a. Pick the input dataset


In [ ]:
## Resolve the latest standardized CSV (run notebook 00 first if missing)
from pathlib import Path
matches = sorted(Path("data/processed").glob("combined_standardized_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
assert matches, "No standardized CSV found in data/processed/. Run 00_prepare_phrasebank_fiqa.ipynb first."
DATA = str(matches[0])
print("DATA =", DATA)


## 3b. Configure parameters


In [ ]:
MAX_ROWS = 0  #@param {type:"integer"}  # 0 = use all rows
SEED = 42  #@param {type:"integer"}
NO_XGBOOST = False  #@param {type:"boolean"}
argv = ["--data", DATA, "--seed", str(SEED)]
if MAX_ROWS: argv += ["--max_rows", str(MAX_ROWS)]
if NO_XGBOOST: argv.append("--no_xgboost")


## 3c. Run `10_run_classical_baselines.py`


In [ ]:
import sys, runpy
sys.argv = ['scripts/10_run_classical_baselines.py'] + argv
print("Running:", " ".join(sys.argv))
runpy.run_path("scripts/10_run_classical_baselines.py", run_name="__main__")


## 4. Inspect latest `classical_baseline_summary_*.csv`


In [ ]:
import pandas as pd
from pathlib import Path
matches = sorted(Path("results").rglob("classical_baseline_summary_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
assert matches, "No summary CSV found — did the script run?"
latest = matches[0]
print("Latest:", latest)
df = pd.read_csv(latest)
df


## ⤓ Download results
Zip `results/` + `figures/` for sharing.


In [ ]:
import shutil, datetime
ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
archive = shutil.make_archive(f"cara_results_{ts}", "zip", root_dir=".", base_dir="results")
print("Created:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass
